# 在 Colab 上训练并评估 LeRobot ACT · pusht

这个笔记本复现本仓库的 ACT / pusht 实验：**LeRobot v0.6.1**，数据集 `lerobot/pusht`（206 集、25650 帧、10 fps），策略 ACT（ResNet18，chunk 100），**一共 100000 步**。依据是 `train.log`、`docs/ANALYSIS_PROCESS.md` 和 `scripts/analyze_training.py`。

按顺序跑下面的单元：

1. **检查 GPU**。没有 CUDA 就不要开训。
2. **可选挂载 Google Drive**，把输出目录指到网盘，断线后 checkpoint 还在。
3. **安装** `lerobot[pusht,training]==0.6.1` 和 `gym-pusht`。训练入口 `lerobot-train` 会检查 `accelerate`，所以必须带上 `training` extra，不能只装 `pusht`。
4. **克隆本仓库**，后面的分析脚本在仓库里。
5. **训练**。超参与 `train.log` 对齐；已有 `checkpoints/last` 时可以续训。
6. **无头评估**。`render_mode=rgb_array`，并把 rollout 写成 mp4，不打开窗口。
7. **跑 `scripts/analyze_training.py --root`** 生成 HTML 报告，并在笔记本里播放一条评估视频。

## 时间和断线

- 选 **GPU** runtime（T4 / L4 / A100）。`train.log` 里预热后大约几十 step/s；T4 上 100K 步常见要 **1.5–3 小时**，另外还要第一次下载 pusht 视频和 ImageNet ResNet18 权重。
- `save_freq=3000`，检查点里有 AdamW 状态。跑满 100K 的检查点体积和仓库里那次实验同一量级（大约十几到二十多 GB）。Colab 本地盘会随会话消失，**要留检查点就打开 Drive**。
- 会话大约 12 小时封顶，空闲也会断。断线后重新打开笔记本，`RESUME = True`，再跑训练单元：它会从 `checkpoints/last` 继续，目标步数仍是 100000。
- **不要提前创建 `OUTPUT_DIR`**。目录已经存在且 `resume=false` 时，`lerobot-train` 会直接报错退出。
- 评估是 50 集、每集 horizon 300。v0.6.1 在 `eval.recording=false`（默认，也是日志里的值）时，把**前 10 集**写成 `{输出目录}/videos/pusht_0/eval_episode_*.mp4`。不要把 `eval.recording` 设成 true：那样会改成写 LeRobot 数据集，并且**不保存 mp4**。
- 本仓库记录的 pusht 成功率是 **0–5%**。loss 收到大约 0.074 并不表示推 T 成功，这是任务本身的结果，不是笔记本坏了。
- `lerobot==0.6.1` 要求 **Python ≥ 3.12**。

## 超参从哪来

`train.log` 的第一段是 `job_name=pusht_act_10k`、`steps=10000`。`docs/ANALYSIS_PROCESS.md` 和 `scripts/analyze_training.py` 写明完整实验是四段续训到 **100000** 步（10K → 50K → 80K → 100K），loss 0.261@10K → 0.074@100K。这个笔记本一次设 `steps=100000`、`job_name=pusht_act_100k`。

和 v0.6.1 默认值不同、并且日志里已经写明的：

| 项 | 日志 | v0.6.1 默认 |
|---|---|---|
| `batch_size` | 16 | 8 |
| `save_freq` | 3000 | 20000 |
| `policy.push_to_hub` | false | **true**。不显式关掉时，缺少 `policy.repo_id` 会拒绝开训 |

其余与日志一致，同时也是 ACT / 训练配置的默认值：`seed=1000`、`log_freq=200`、`num_workers=4`、`env_eval_freq=20000`、`eval.n_episodes=50`、`eval.batch_size=22`、`wandb.enable=false`、`dataset.use_imagenet_stats=true`、chunk / `n_action_steps` = 100、`n_obs_steps=1`、ResNet18、`dim_model=512`、8 heads、`dim_feedforward=3200`、encoder 4 层、decoder 1 层、VAE（`latent_dim=32`、`kl_weight=10`）、`dropout=0.1`、策略和 backbone 学习率都是 `1e-5`、`weight_decay=1e-4`、`use_policy_training_preset=true`。

日志开头是 `Device 'None' is not available. Switching to 'cuda'.`：当时没有传 `policy.device`，运行时自动选了 CUDA。有 GPU 时这里显式传 `--policy.device=cuda`。训练配置里的 `env` 是空的，所以训练循环不做仿真评估；评估单独用 `lerobot-eval`。


## 1. 检查 GPU

需要 CUDA。若打印 `cuda False`，菜单 **Runtime → Change runtime type → GPU**，然后再跑这一格。


In [ ]:
import torch

cuda = torch.cuda.is_available()
print("python", __import__("sys").version.split()[0])
print("torch", torch.__version__)
print("cuda", cuda)
if cuda:
    print("device", torch.cuda.get_device_name(0))
else:
    print("没有 GPU。训练单元会停下来。请换成 GPU runtime 后再继续。")


## 2. 可选：挂载 Google Drive

默认不挂载，检查点写在 Colab 本地盘，断线即丢失。把 `USE_DRIVE` 改成 `True` 后，输出目录和 Hugging Face 缓存都放在 `MyDrive/robot/`，下次打开笔记本可以接着训。

`RESUME = True` 时，如果该目录下已有 `checkpoints/last`，训练单元会续训，而不是从头再来。


In [ ]:
USE_DRIVE = False  # 改成 True：检查点写到 Google Drive
RESUME = True      # 已有 checkpoints/last 时从该检查点续训

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive 已挂载。输出将写到 /content/drive/MyDrive/robot/")
else:
    print("未挂载 Drive。检查点只在本次 Colab 会话的本地盘上。")


## 3. 安装 LeRobot v0.6.1 和 gym-pusht

版本钉在与本仓库相同的 **0.6.1**。`pusht` extra 带上数据集解码和 `gym-pusht`；`training` extra 提供 `lerobot-train` 需要的 `accelerate`。`gym-pusht` 再显式装一次，约束与 LeRobot 0.6.1 的 extra 一致（`>=0.1.5,<0.2.0`）。

安装会升级 Colab 预装的 PyTorch，常常要几分钟。如果装完提示必须重启 runtime，重启后从**下一格（克隆仓库）**继续即可，不必重装。


In [ ]:
import sys
import subprocess

print(sys.version)
if sys.version_info < (3, 12):
    raise SystemExit("lerobot 0.6.1 需要 Python >= 3.12。请更换 Python 3.12 的 runtime。")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "lerobot[pusht,training]==0.6.1",
    "gym-pusht>=0.1.5,<0.2.0",
    "matplotlib",
])

import importlib.metadata as metadata
import lerobot
print("lerobot", lerobot.__version__)
print("gym-pusht", metadata.version("gym-pusht"))


## 4. 克隆本仓库

分析脚本 `scripts/analyze_training.py`、历史 `train.log` 和 MimicGen 日志都在仓库里。Colab 从徽章打开笔记本时，只会拿到笔记本本身，所以这里要再克隆一次。


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/skyxiaobai/robot.git"
REPO_DIR = "/content/robot"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
else:
    print("已有克隆，跳过:", REPO_DIR)

os.environ["ROBOT_ROOT"] = REPO_DIR
print("ROBOT_ROOT", os.environ["ROBOT_ROOT"])


## 5. 训练 ACT

输出目录在这一格才决定：挂了 Drive 就写到网盘，否则写到仓库下的 `outputs/train/pusht_act_100k`。Hugging Face 缓存放在输出目录的**旁边**，避免先把 `OUTPUT_DIR` 建出来（那样训练会拒绝启动）。

续训命令是 v0.6.1 的真实接口：

```bash
lerobot-train --resume=true \
  --config_path=$OUTPUT_DIR/checkpoints/last/pretrained_model/train_config.json
```

检查点里的 `steps` 已经是 100000，续训会从上次的 step 跑到 100000。第一次训练的 stdout 会覆盖 `outputs/colab_train.log`；续训则追加。分析脚本在本机没有 `/tmp/train_50k.log` 时，用这份日志画曲线（按 `log_freq=200` 数行，和原来的 `train.log` 同一套算法）。

只想确认命令能跑通时，把 `STEPS` 改成 `20` 之类再跑；正式复现保持 `100000`。`NUM_WORKERS` 与日志一样是 4。双核 Colab 上如果 DataLoader 子进程被杀掉，改成 2。


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import torch

STEPS = 100_000
NUM_WORKERS = 4
device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    raise SystemExit("需要 GPU runtime。请先换 GPU，再重跑检查 GPU 的那一格。")

use_drive = bool(globals().get("USE_DRIVE", False))
resume_flag = bool(globals().get("RESUME", True))
if use_drive:
    base = Path("/content/drive/MyDrive/robot")
    OUTPUT_DIR = str(base / "pusht_act_100k")
    HF_HOME = str(base / "hf_home")
    HF_LEROBOT_HOME = str(base / "hf_lerobot_home")
else:
    OUTPUT_DIR = str(Path(REPO_DIR) / "outputs" / "train" / "pusht_act_100k")
    HF_HOME = "/content/hf_home"
    HF_LEROBOT_HOME = "/content/hf_lerobot_home"

out = Path(OUTPUT_DIR)
last_cfg = out / "checkpoints" / "last" / "pretrained_model" / "train_config.json"
resuming = bool(resume_flag and last_cfg.is_file())
if not resuming and out.exists():
    raise SystemExit(
        f"{out} 已存在，且没有 checkpoints/last。"
        "lerobot-train 在 resume=false 时不会覆盖已有目录。"
        "删掉该目录，或确认 RESUME=True 且检查点在，或换一个 OUTPUT_DIR。"
    )

cmd = ["lerobot-train"]
if resuming:
    cmd += ["--resume=true", f"--config_path={last_cfg}"]
    print("从检查点续训:", last_cfg)
else:
    cmd += [
        "--dataset.repo_id=lerobot/pusht",
        "--policy.type=act",
        f"--policy.device={device}",
        "--policy.push_to_hub=false",
        "--batch_size=16",
        f"--steps={STEPS}",
        "--save_freq=3000",
        "--save_checkpoint=true",
        f"--num_workers={NUM_WORKERS}",
        "--log_freq=200",
        "--seed=1000",
        "--job_name=pusht_act_100k",
        "--wandb.enable=false",
        "--dataset.use_imagenet_stats=true",
        "--env_eval_freq=20000",
        "--eval.n_episodes=50",
        "--eval.batch_size=22",
        "--policy.chunk_size=100",
        "--policy.n_action_steps=100",
        "--policy.n_obs_steps=1",
        "--policy.vision_backbone=resnet18",
        "--policy.pretrained_backbone_weights=ResNet18_Weights.IMAGENET1K_V1",
        "--policy.dim_model=512",
        "--policy.n_heads=8",
        "--policy.dim_feedforward=3200",
        "--policy.n_encoder_layers=4",
        "--policy.n_decoder_layers=1",
        "--policy.use_vae=true",
        "--policy.latent_dim=32",
        "--policy.n_vae_encoder_layers=4",
        "--policy.kl_weight=10.0",
        "--policy.dropout=0.1",
        "--policy.optimizer_lr=1e-5",
        "--policy.optimizer_lr_backbone=1e-5",
        "--policy.optimizer_weight_decay=1e-4",
        "--use_policy_training_preset=true",
        f"--output_dir={out}",
    ]
    print("从头训练，输出目录:", out)

env = os.environ.copy()
env["HF_HOME"] = HF_HOME
env["HF_LEROBOT_HOME"] = HF_LEROBOT_HOME
env["SDL_VIDEODRIVER"] = "dummy"
env["SDL_AUDIODRIVER"] = "dummy"

log_path = Path(REPO_DIR) / "outputs" / "colab_train.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
print("日志:", log_path)
print(" ".join(cmd))

with log_path.open("a" if resuming else "w") as log, subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
    cwd=REPO_DIR,
) as proc:
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        log.write(line)
        log.flush()
    code = proc.wait()
if code != 0:
    raise SystemExit(f"lerobot-train 退出码 {code}")
print("训练结束:", out)


## 6. 无头评估，并把 rollout 存成视频

`lerobot-eval` 用 `rgb_array` 渲染，视频写到磁盘，不调用 `cv2.imshow`。进程环境里设置 `SDL_VIDEODRIVER=dummy`，避免 pygame 在没有显示器时去开窗口。

与 `train.log` 里评估配置一致的是 `n_episodes=50`、`batch_size=22`（日志里的 22 是当时机器上解析出来的值）。每集 `episode_length=300`，对应报告里的 horizon 300。前 10 集的 mp4 在 `videos/pusht_0/`。50 集在 GPU 上通常还要几十分钟。

评估目录如果在 Drive 上，这一格会把 `eval_info.json` 复制到仓库的 `outputs/eval/colab/`，让下一步的分析脚本能读到。


In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

import torch

EVAL_EPISODES = 50
EVAL_BATCH_SIZE = 22
device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    raise SystemExit("评估也放在 GPU runtime 上跑。")

use_drive = bool(globals().get("USE_DRIVE", False))
if use_drive:
    EVAL_DIR = "/content/drive/MyDrive/robot/eval_pusht_act"
else:
    EVAL_DIR = str(Path(REPO_DIR) / "outputs" / "eval" / "colab")

ckpt = Path(OUTPUT_DIR) / "checkpoints" / "last" / "pretrained_model"
if not (ckpt / "config.json").is_file():
    numbered = sorted(Path(OUTPUT_DIR).glob("checkpoints/*/pretrained_model/config.json"))
    if not numbered:
        raise SystemExit(f"在 {OUTPUT_DIR} 没找到 checkpoint。先跑训练单元。")
    ckpt = numbered[-1].parent

cmd = [
    "lerobot-eval",
    f"--policy.path={ckpt}",
    "--env.type=pusht",
    "--env.task=PushT-v0",
    "--env.obs_type=pixels_agent_pos",
    "--env.render_mode=rgb_array",
    "--env.episode_length=300",
    f"--eval.n_episodes={EVAL_EPISODES}",
    f"--eval.batch_size={EVAL_BATCH_SIZE}",
    "--eval.recording=false",
    f"--policy.device={device}",
    "--policy.use_amp=false",
    "--seed=1000",
    f"--output_dir={EVAL_DIR}",
]
print(" ".join(cmd))

env = os.environ.copy()
env["SDL_VIDEODRIVER"] = "dummy"
env["SDL_AUDIODRIVER"] = "dummy"
env.setdefault("HF_HOME", HF_HOME)
env.setdefault("HF_LEROBOT_HOME", HF_LEROBOT_HOME)

with subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
    cwd=REPO_DIR,
) as proc:
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
    code = proc.wait()
if code != 0:
    raise SystemExit(f"lerobot-eval 退出码 {code}")

src = Path(EVAL_DIR) / "eval_info.json"
dest = Path(REPO_DIR) / "outputs" / "eval" / "colab" / "eval_info.json"
if src.is_file() and src.resolve() != dest.resolve():
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dest)
    print("已复制 eval_info.json 到", dest)
print("评估视频目录:", Path(EVAL_DIR) / "videos")


## 7. 生成报告，并播放一条 rollout

`scripts/analyze_training.py` 读取 `--root`（未传时用环境变量 `ROBOT_ROOT`，克隆那一格已经设好）。它会写出 `outputs/report_20260801/training_report.html`。

在 Colab 上，曲线优先用训练单元写下的 `outputs/colab_train.log`（本机没有原来的 `/tmp/train_50k.log` 时）。评估图读取 `outputs/eval/**/eval_info.json`，包括上一格复制过来的结果。仓库里已有的 `train.log` 和 MimicGen 日志仍会在没有 Colab 训练日志时被用到。

横轴按 `log_freq=200` 数日志行，和原始实验同一约定。请保持训练命令里的 `--log_freq=200`。


In [ ]:
import subprocess
import sys
from pathlib import Path

from IPython.display import Video, display

# 图表里的中文。没有 root 或字体包时，报告仍会生成，只是坐标轴可能缺字。
subprocess.call(["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"])
subprocess.call(["fc-cache", "-f"])

subprocess.check_call(
    [sys.executable, "scripts/analyze_training.py", "--root", REPO_DIR],
    cwd=REPO_DIR,
)

report = Path(REPO_DIR) / "outputs" / "report_20260801" / "training_report.html"
print("报告:", report)

videos = sorted(Path(EVAL_DIR).rglob("eval_episode_*.mp4"))
if not videos:
    raise SystemExit(f"没有在 {EVAL_DIR} 找到 eval_episode_*.mp4")
print("播放", videos[0])
display(Video(str(videos[0]), embed=True, width=384))
